PePy

Author: Julia K. Varga <jvarga92@gmail.com>  
License: BSD 3 clause  
Code Repository: https://github.com/gezmi/pepy

In [1]:
import pandas as pd
pd.set_option('display.width', 600)
pd.set_option('display.max_columns', 8)

# Working with Protein Complexes

`ProteinComplex` is the main class in PePy. It extends BioPandas' `PandasPdb` with chain assignment, interface calculation, and confidence metrics.

This tutorial covers loading structures, identifying chains, calculating interfaces, and accessing the resulting data as DataFrames.

In [2]:
from pepy import ProteinComplex

## Loading Structures

PePy accepts `.pdb` and `.cif` files. The easiest way is the `from_file` class method:

In [3]:
cx = ProteinComplex.from_file('../../pepy/tests/data/1YCR.pdb')
cx.df['ATOM'].head()

,record_name,atom_number,blank_1,atom_name,...,element_symbol,charge,line_idx,ca_index
0,ATOM,1,,N,...,N,NaN,339,0
1,ATOM,2,,CA,...,C,NaN,340,0
2,ATOM,3,,C,...,C,NaN,341,0
3,ATOM,4,,O,...,O,NaN,342,0
4,ATOM,5,,CB,...,C,NaN,343,0


You can also load CIF files — AF3 CIF files with missing columns are fixed automatically:

In [4]:
cx_cif = ProteinComplex.from_file('../../pepy/tests/data/fold_1ycr_af3_model_0.cif')
cx_cif.df['ATOM'].head()

,record_name,atom_number,blank_1,atom_name,...,element_symbol,charge,line_idx,ca_index
0,ATOM,1,,N,...,N,NaN,0,0
1,ATOM,2,,CA,...,C,NaN,1,0
2,ATOM,3,,C,...,C,NaN,2,0
3,ATOM,4,,O,...,O,NaN,3,0
4,ATOM,5,,CB,...,C,NaN,4,0


Or fetch directly from the PDB:

In [5]:
cx_fetch = ProteinComplex()
cx_fetch.fetch_pdb('1ycr')
print('Chains:', cx_fetch.get_available_chains())

Chains: ['A', 'B']


## Inspecting the Structure

Before assigning chains, it's useful to see what's in the file:

In [6]:
cx = ProteinComplex.from_file('../../pepy/tests/data/1ycr_af2_55d19_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000.pdb')

print('Available chains:', cx.get_available_chains())
print('Chain lengths:', cx.get_chain_lengths())

Available chains: ['A', 'B']
Chain lengths: {'A': 109, 'B': 15}


## Identifying Chains

You need to tell PePy which chain(s) are the **binder** and which are the **receptor** before calculating the interface.

### Auto-detection

By default, the shortest chain becomes the binder:

In [7]:
cx.identify_chains()

print(f'Binder:   {cx.binder_chains}')
print(f'Receptor: {cx.receptor_chains}')

Binder chain(s): B, receptor chain(s): A
Binder:   ['B']
Receptor: ['A']


### Explicit assignment

You can specify the binder as a string (single chain) or list (multi-chain):

In [8]:
# Single-chain binder
cx.identify_chains(binder_chains='B', receptor_chains=['A'])

# Multi-chain binder (if applicable)
# cx.identify_chains(binder_chains=['B', 'C'], receptor_chains=['A'])

Binder chain(s): B, receptor chain(s): A


### Chain info

Get detailed information about all chains — lengths, roles, sequences:

In [9]:
info = cx.get_chain_info()
for chain_id, chain_data in info.items():
    print(f"Chain {chain_id}: {chain_data['length']} residues, "
          f"role={chain_data['type']}, seq={chain_data['sequence'][:30]}...")

Chain A: 109 residues, role=receptor, seq=SQIPASEQETLVRPKPLLLKLLKSVGAQKD...
Chain B: 15 residues, role=binder, seq=SQETFSDLWKLLPEN...


## Calculating the Interface

The interface calculation uses a two-stage algorithm:

1. **CB prefiltering** — finds residues whose CB atoms (CA for glycine) are within `cb_cutoff` Å
2. **All-atom refinement** — among those, finds residues with any atom within `all_atom_cutoff` Å

Either stage can be disabled by setting its cutoff to `-1`.

In [10]:
binder_res, receptor_res = cx.calculate_interface(
    cb_cutoff=8.0,          # Å, CB prefilter
    all_atom_cutoff=4.0,    # Å, all-atom refinement
)

print(f'Binder interface residues:   {binder_res}')
print(f'Receptor interface residues: {receptor_res}')

Binder interface residues:   [2, 3, 4, 5, 8, 9, 11, 12, 13, 14]
Receptor interface residues: [35, 38, 42, 45, 51, 54, 56, 57, 59, 77, 80, 83, 84]


### Optional: confidence filtering

You can filter out low-confidence binder residues based on their pLDDT (stored in the B-factor column):

In [11]:
binder_res_filtered, receptor_res_filtered = cx.calculate_interface(
    drop_low_confidence=True,
    confidence_threshold=70.0,
)

print(f'Before filtering: {len(binder_res)} binder residues')
print(f'After filtering:  {len(binder_res_filtered)} binder residues')

Before filtering: 10 binder residues
After filtering:  10 binder residues


### Interface summary

In [12]:
cx.calculate_interface()  # recalculate without filtering
cx.get_interface_summary()

{'status': 'calculated',
 'binder_interface_residues': 10,
 'receptor_interface_residues': 13,
 'binder_interface_atoms': 90,
 'receptor_interface_atoms': 113,
 'binder_residue_list': [2, 3, 4, 5, 8, 9, 11, 12, 13, 14],
 'receptor_residue_list': [35, 38, 42, 45, 51, 54, 56, 57, 59, 77, 80, 83, 84]}

## Accessing Interface Atoms

After calculating the interface, you can get the actual atoms as pandas DataFrames — useful for downstream analysis, visualization, or export.

In [13]:
# All interface atoms
binder_atoms, receptor_atoms = cx.get_interface_atoms('all')
print(f'Binder interface atoms:   {len(binder_atoms)}')
print(f'Receptor interface atoms: {len(receptor_atoms)}')

binder_atoms.head()

Binder interface atoms:   90
Receptor interface atoms: 113


,record_name,atom_number,blank_1,atom_name,...,element_symbol,charge,line_idx,ca_index
885,ATOM,887,,N,...,N,NaN,887,110
886,ATOM,888,,CA,...,C,NaN,888,110
887,ATOM,889,,C,...,C,NaN,889,110
888,ATOM,890,,CB,...,C,NaN,890,110
889,ATOM,891,,O,...,O,NaN,891,110


In [14]:
# Filter by atom type: 'ca', 'backbone', 'sidechain'
binder_ca, receptor_ca = cx.get_interface_atoms('ca')
binder_bb, receptor_bb = cx.get_interface_atoms('backbone')
binder_sc, receptor_sc = cx.get_interface_atoms('sidechain')

print(f'CA only:    {len(binder_ca)} + {len(receptor_ca)}')
print(f'Backbone:   {len(binder_bb)} + {len(receptor_bb)}')
print(f'Sidechain:  {len(binder_sc)} + {len(receptor_sc)}')

CA only:    10 + 13
Backbone:   40 + 52
Sidechain:  50 + 61


These are standard pandas DataFrames — you can select columns, merge, export to CSV, etc.:

In [15]:
binder_ca[['chain_id', 'residue_number', 'residue_name', 'x_coord', 'y_coord', 'z_coord', 'b_factor']]

,chain_id,residue_number,residue_name,x_coord,y_coord,z_coord,b_factor
886,B,2,GLN,-1.901,-20.938,6.602,83.00
895,B,3,GLU,-1.017,-17.328,7.191,91.62
904,B,4,THR,-3.686,-15.188,5.379,96.94
911,B,5,PHE,-2.848,-12.141,3.359,98.19
936,B,8,LEU,0.399,-10.672,6.918,97.38
944,B,9,TRP,-0.491,-7.168,5.688,97.94
967,B,11,LEU,2.086,-6.809,10.344,95.44
975,B,12,LEU,3.389,-3.918,8.250,95.31
983,B,13,PRO,4.879,-1.167,10.570,86.56
990,B,14,GLU,2.395,1.711,10.766,74.44


## Clearing and Recalculating

Re-assigning chains automatically clears previous interface results:

In [16]:
print('Interface calculated:', cx.interface_calculated)

# Swap binder and receptor
cx.identify_chains(binder_chains='A', receptor_chains=['B'])
print('After re-identifying chains:', cx.interface_calculated)

# Recalculate with new assignment
binder_res, receptor_res = cx.calculate_interface()
print(f'Now binder (chain A) has {len(binder_res)} interface residues')

Interface calculated: True
Binder chain(s): A, receptor chain(s): B
After re-identifying chains: False


Now binder (chain A) has 13 interface residues


You can also clear results manually:

In [17]:
cx.clear_interface_results()
print('Interface calculated:', cx.interface_calculated)

cx.clear_cache()  # clears everything including chain length cache

Interface calculated: False
